# 02. 오프라인 평가와 Golden Dataset

> Part 2. 무엇을 성공으로 볼지 어떻게 고정하는가

01에서 그럴듯한 답이 업무 성공의 증거가 못 된다는 것을 봤습니다. 이 노트북은 그다음 질문에 답합니다. **성공의 기준을 누가, 무엇에 근거해, 어떻게 고정하고, 고정한 기준으로 에이전트를 어떻게 채점하는가.**

## 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 준비 | 키 상태와 산출물 폴더, Langfuse 연결 확인 | 3분 |
| Scenario 계약 | 입력, 평가자 전용 기준, 꼬리표를 나눠 읽기 | 5분 |
| 사례를 얻는 세 경로 | 담당자 인터뷰, 정책 문서, 운영 trace | 8분 |
| 문서 기반 합성 후보 | 정책 문서에서 후보를 만들고 사람이 승인한다 | 6분 |
| 최소쌍과 커버리지 | 한 조건만 바꾼 쌍, family_id와 slice 표 | 6분 |
| 검수·동결·승인 게이트 | 동결 해시, dev와 holdout 분할, 담당자 승인 전의 holdout | 8분 |
| 실험 실행과 Dataset | dev 3사례 x 2회 실행, Langfuse Dataset과 실험 run 기록 | 7분 |
| 판정자 넷 | Code, Jev, Judge, Human, 연쇄와 사람 라벨 대조 | 12분 |
| 직접 해보기 | 라벨 쓰기, 후보 만들기, trace 승격, 문턱값 탐색 | 5분 |

## 이 노트북이 다루는 것

1. 사례 카드 하나(Scenario)가 에이전트에게 주는 것과 평가자에게만 주는 것을 나눈다.
2. 사례를 얻는 세 경로(인터뷰, 문서, 운영 trace)와 각각의 약점을 본다.
3. 문서에서 후보를 만들 때 정답을 모델이 쓰게 하지 않고, 사람이 승인하는 흐름을 실제 엑셀 왕복으로 확인한다.
4. 한 조건만 바꾼 최소쌍과 커버리지 표로 빠진 능력을 찾는다.
5. 검수와 동결 해시, dev와 holdout 분할, 담당자 승인 전에는 holdout이 열리지 않는 게이트를 직접 부딪쳐 본다.
6. 동결한 사례를 실제 모델로 돌리고, 실행 기록을 Langfuse Dataset과 실험 run 이름으로 남긴다.
7. 판정을 Code, Jev, Judge, Human 네 역할로 나누고, 의미 판정의 세 방식(연쇄, Jev 단독, Judge 단독)을 사람 라벨과 대조한다.

## 호출 규모

이 노트북의 유료 셀은 두 군데이고, 각 셀은 실행 직전에 `announce`로 규모를 한 줄로 알립니다. 아래는 기본 실행의 상한입니다. 측정값이 아니라 예고 값입니다.

| 셀 | 에이전트 모델 | Jev | Judge(xhigh) |
|---|---|---|---|
| dev 3사례 x 2회 실행 | 최대 72 | 0 | 0 |
| 연쇄(cascade) 판정, 같은 6 Episode | 0 | 6 | 최대 18 |
| Jev 단독과 Judge 단독 비교, 같은 6 Episode | 0 | 6 | 최대 18 |

Judge는 질문마다 한 번 부르므로 Episode 하나에 질문 세 개면 Judge 상한은 Episode 수의 세 배입니다. 기본 실행 합계는 에이전트 최대 72회, Jev 12회, Judge 최대 36회입니다. 직접 해보기와 선택 실습의 스위치(`RUN_EXERCISE`, `RUN_OPTIONAL`)는 유료 호출을 더하지 않습니다.

Langfuse 키가 없어도 끝까지 갑니다. 그때는 Dataset 적재와 점수 붙이기를 건너뛰고 같은 내용을 로컬 파일에 남깁니다.

모든 사례는 교육용 가상 데이터입니다. 실제 쇼핑몰 정책이나 법률 안내가 아닙니다.

## 0. 오프라인 평가는 무엇을 하는가

오프라인 평가는 **정답에 해당하는 기준을 미리 정해 두고**, 그 기준으로 에이전트의 여러 버전을 같은 조건에서 비교하는 일입니다. 운영에서 들어오는 요청에는 정해 둔 정답이 없습니다. 그래서 운영 trace를 채점하는 온라인 평가는 03에서 따로 다룹니다. 오프라인 평가가 하는 일은 세 가지로 나눌 수 있습니다.

| 하는 일 | 질문 | 이 노트북에서 |
|---|---|---|
| 기준을 고정한다 | 무엇이 끝나면 성공이고, 무엇을 하면 실패인가 | Scenario, 인터뷰, 정책 문서, 검수, 동결 |
| 비교 가능하게 만든다 | 두 버전이 같은 사례·같은 예산·같은 반복으로 돌았는가 | dev와 holdout 분할, 동결 해시, 실험 run 이름 |
| 판정자를 믿을 수 있는지 잰다 | 자동 판정이 사람이 보는 것과 얼마나 어긋나는가 | Code, Jev, Judge, Human, 사람 라벨 대조 |

기준을 고정하지 않고 에이전트를 돌려 보면 숫자는 나오지만 그 숫자가 무엇을 뜻하는지 말할 수 없습니다. 이 노트북의 순서는 "기준 → 동결 → 실행 → 판정 → 판정자 점검"입니다.

In [ ]:
# 스위치: 직접 해보기와 선택 실습을 켜는 값이다. 기본은 둘 다 꺼 둔다.
# 두 값 모두 유료 호출을 더하지 않는다. RUN_EXERCISE는 직접 해보기의 라벨 대조와 trace 승격을, RUN_OPTIONAL은 문턱값 격자를 켠다.
RUN_EXERCISE = False
RUN_OPTIONAL = False

In [ ]:
# [따라하기] 1단계: 도구를 불러오고 키 상태, 산출물 폴더, Langfuse 연결을 확인한다
import json
import re
from collections import Counter
from copy import deepcopy

from eval_lab.agent import PROMPTS
from eval_lab.authoring import (
    draft_cases,
    freeze,
    load_frozen,
    read_workbook,
    write_workbook,
)
from eval_lab.contracts import Scenario
from eval_lab.dataset import (
    DATA,
    POLICY_VERSION,
    fingerprint,
    load_dataset,
    load_fixtures,
    pending_ids,
)
from eval_lab.decisions import make_decider
from eval_lab.episodes import execute_episode
from eval_lab.evaluators import evaluate
from eval_lab.evidence import promote_failure
from eval_lab.judges import T_HI, T_LO, JudgeUsage, judge_semantics
from eval_lab.notebook import announce, langfuse_or_none, new_run_dir, show, show_keys
from eval_lab.runs import check_episode
from eval_lab.settings import load_settings, make_agent_model, make_judge_model
from eval_lab.storage import make_run_id, save_run_report
from pydantic import ValidationError

settings = load_settings()
show_keys(settings)          # 값은 보여 주지 않고 설정됨/미설정만 보여 준다

client = langfuse_or_none(settings)   # 키가 없으면 안내하고 None을 돌려준다
run_dir = new_run_dir("02")           # 이 노트북이 쓰는 파일은 모두 이 폴더 안에 둔다
print("Langfuse:", "연결됨" if client is not None else "없음(로컬 기록으로 진행)")
print("산출물 폴더:", run_dir.name)

키 상태는 이름과 "설정됨/미설정"만 보입니다. 값은 어디에도 출력하지 않습니다. 이 노트북에서 모델을 쓰려면 `OPENGATEWAY_API_KEY`(에이전트와 Judge)와 Decision API 키(Jev)가 필요하고, Langfuse 키는 있으면 쓰고 없으면 로컬 경로로 갑니다.

## 1. Scenario 계약: 입력, 평가자 전용 기준, 꼬리표

사례 카드를 "질문과 답의 표"로 만들면 에이전트가 무엇을 했는지 볼 수 없습니다. 이 저장소의 사례 카드(`Scenario`)는 세 덩어리로 나뉩니다.

| 덩어리 | 누가 보나 | 담는 것 |
|---|---|---|
| `input` + `fixture_id` | 에이전트와 실행 하네스 | 고객의 요청, 주문 번호, 시작 상태를 만드는 fixture의 ID |
| `expected_output` | **평가자만** | 기대 최종 상태, 금지 도구, 기대 응답 종류, 환불 시도 한도, 필수 이정표 |
| `metadata` | 추적·분류용 | 사례 ID, family_id, slice, split, 출처, 정책 버전, 검수 상태, 검수자와 사유 |

핵심은 실제 결과(output)와 기대 결과(expected_output)가 **다른 필드**라는 점입니다. 에이전트의 출력은 관측값이고, `expected_output`은 사람이 검수해 정한 별도의 기준입니다. 에이전트가 낸 결과를 기대 상태로 복사하면 평가가 아니라 자기 확인이 됩니다.

계약은 `eval_lab/contracts.py`의 `Scenario`입니다. 모든 계약은 `extra="forbid"`, `strict=True`라서 모르는 필드와 타입이 다른 값을 거부합니다.

In [ ]:
# [따라하기] 2단계: 사례 카드 한 장을 에이전트에게 주는 것, 평가자만 보는 것, 꼬리표로 나눠 본다
dev_cards = load_dataset(split="dev")                       # 검수가 끝난 dev 사례만 읽는다
cards_by_id = {card.metadata.scenario_id: card for card in dev_cards}
card = cards_by_id["normal"]

print("[에이전트에게 주는 입력]")
print(card.input.model_dump())
print("fixture_id:", card.fixture_id)
print()
print("[평가자만 보는 기준]")
print(card.expected_output.model_dump())
print()
print("[꼬리표]")
print(card.metadata.model_dump())

**먼저 예측하세요.** 같은 요청 문장("이 주문을 취소하고 결제 금액을 환불해 주세요.")에 주문 상태만 `shipped`(출고 완료)로 바꾼 카드는, 위 카드와 비교해 **어떤 필드가** 달라질까요? 입력 문장, fixture, 기대 상태, 금지 도구, 기대 응답 종류 중에서 골라 보세요. 아래 셀에서 확인합니다.

In [ ]:
# [바꿔보기] 3단계: CARD_IDS를 바꿔 사례마다 평가 기준이 어떻게 다른지 표로 비교한다
CARD_IDS = ["normal", "shipped", "missing-id"]   # dev 사례 ID 중에서 골라 바꿔 본다
selected = [cards_by_id[scenario_id] for scenario_id in CARD_IDS]
show([
    {
        "scenario_id": item.metadata.scenario_id,
        "slice": item.metadata.slice,
        "입력 order_id": item.input.order_id,
        "기대 응답 종류": item.expected_output.response_kind,
        "금지 도구 수": len(item.expected_output.forbidden_tools),
        "환불 시도 한도": item.expected_output.max_refund_attempts,
        "기대 최종 상태": item.expected_output.final_state,
    }
    for item in selected
])

**관찰할 것.** `normal`과 `shipped`는 요청 문장이 같은데 기대 응답 종류와 금지 도구가 다릅니다. 요청 문장이 같아도 **주문의 상태**(fixture)가 다르면 정답 행동이 달라집니다. 그래서 사례 카드는 문장이 아니라 "초기 조건 + 기대 상태 + 행동 제약"의 묶음이어야 합니다. `missing-id`는 입력 자체에 주문 번호가 없고, 기대 응답이 `clarification`(되묻기)입니다. 거절과 되묻기도 사례로 넣어야 하는 이유는 이렇습니다. 정상 처리만 있으면 무조건 실행하는 에이전트가 가장 높은 점수를 받습니다.

In [ ]:
# [깨뜨리기] 4단계: 기대 최종 상태를 비운 카드와 모르는 필드가 있는 카드를 계약이 거부하는지 본다
empty_state = card.model_dump()
empty_state["expected_output"]["final_state"] = {}          # "기대 상태 없음"으로 만든다
extra_field = card.model_dump()
extra_field["expected_output"]["reference"] = "에이전트 답을 그대로 베낀 값"   # 계약에 없는 필드

for label, broken in [("기대 상태 비움", empty_state), ("계약에 없는 필드", extra_field)]:
    try:
        Scenario.model_validate(broken)
    except ValidationError as error:
        first = error.errors()[0]
        print(f"{label}: 거부됨 -> {first['loc']} / {first['type']}")
    else:
        print(f"{label}: 통과 (이 줄이 보이면 계약이 느슨해진 것입니다)")

**관찰할 것.** 두 카드 모두 거부됩니다. `final_state`가 비면 "아무 상태나 통과"가 되어 평가가 의미를 잃으므로 계약이 길이 1 이상을 요구합니다. 모르는 필드도 막습니다. 평가자가 읽는지 아닌지 모를 필드가 슬쩍 섞여 들어오는 일을 막기 위해서입니다. 평가 기준의 오류는 에이전트의 실패로 둔갑하면 안 되므로, 이런 카드는 **채점 전에** 거부합니다.

## 2. 사례를 얻는 세 경로

| 경로 | 알려 주는 것 | 약점 |
|---|---|---|
| 업무 담당자 인터뷰 | **기준**: 무엇이 끝나야 완료인가, 무엇은 절대 하면 안 되는가 | 담당자의 기억에 의존하고, 말로 한 기준은 판정 가능한 조건이 아니다 |
| 업무 문서(정책·SOP·FAQ) | **범위**: 조건과 경계값, 금지와 예외 | 문서가 정답을 대신하지 않는다. 문서에 없는 예외는 문서만 보고 만들 수 없다 |
| 운영 trace | **빠진 실패**: 오류, 불만, 희귀 요청 | 재현 가능해야 하고, 개인정보를 지워야 하고, 기대 상태를 사람이 다시 정해야 한다 |

세 경로는 서로를 보완합니다. 인터뷰는 기준을 주고, 문서는 범위를 채우고, 운영 trace는 앞의 두 경로가 놓친 빈틈을 알려 줍니다.

### 2.1 담당자 인터뷰: 말을 판정 가능한 조건으로 바꾼다

인터뷰 원문은 `data/interview.md`에 있습니다. 매 질문마다 "어떤 과정이 있으면 성공이라고 판정할 수 있나요?"를 되묻는 것이 핵심입니다. "고객에게 친절하게"는 판정할 수 없지만 "실제 확인한 완료·미완료만 말한다"는 기록과 대조할 수 있습니다.

In [ ]:
# [따라하기] 5단계: 인터뷰 표에서 질문, 담당자의 답, 명세할 것을 읽어 온다
interview_lines = (DATA / "interview.md").read_text(encoding="utf-8").splitlines()
table_rows = [[cell.strip() for cell in line.strip("|").split("|")] for line in interview_lines if line.startswith("|")]
interview = [{"질문": row[0], "담당자의 답": row[1], "명세할 것": row[2]} for row in table_rows[2:]]   # 앞 두 줄은 머리글과 구분선
show(interview)
print("질문 수:", len(interview))

In [ ]:
# [바꿔보기] 6단계: 인터뷰 질문 하나하나가 Scenario의 어느 필드가 되고, 어느 dev 사례가 그것을 시험하는지 잇는다
INTERVIEW_TO_CARD = {
    "무엇이 끝나면 취소 완료인가요?": ("expected_output.final_state", "normal"),
    "언제 처리하면 안 되나요?": ("forbidden_tools와 보호 상태", "foreign-order"),
    "주문 번호가 없으면요?": ("response_kind = clarification", "missing-id"),
    "결제 API가 TIMEOUT이면요?": ("결과 미확정 -> 원장 조회 후 재시도", "timeout-before"),
    "이미 환불됐으면요?": ("추가 환불 금지(max_refund_attempts)", "timeout-after"),
    "재시도해도 해결이 안 되면요?": ("부분 완료 보존과 담당자 이관", "payment-down"),
    "고객에게 무엇을 말하나요?": ("응답 문장의 의미(Jev, Judge 몫)", "normal"),
}
asked = {row["질문"] for row in interview}
show([
    {
        "인터뷰 질문": question,
        "되는 필드": field,
        "시험하는 dev 사례": f"{scenario_id} ({cards_by_id[scenario_id].metadata.slice})",
        "인터뷰에 있는 질문": question in asked,
    }
    for question, (field, scenario_id) in INTERVIEW_TO_CARD.items()
])

인터뷰의 마지막 질문("고객에게 무엇을 말하나요?")에 주목하세요. 이 답은 구조화된 필드(`kind`, 주문 번호, 환불 금액)가 아니라 **자유 문장**에 대한 기준입니다. 구조화 필드는 실제 상태와 코드로 대조할 수 있지만, 문장이 기록보다 부풀렸는지는 의미 판정이 필요합니다. 이 부분이 뒤의 Jev와 Judge가 맡는 자리입니다.

### 2.2 정책 문서: 조항마다 시험할 조건을 후보로 적는다

정책 원문은 `data/order_policy.md`이고 기준일(as_of)은 2026-09-10, 정책 버전은 `order-policy-v1`입니다. 문서에서 후보를 만들 때의 규칙은 하나입니다. **후보마다 근거 문단(원문 인용)과 정책 버전을 붙인다.** 아래 셀은 조항(P1~P5)을 쪼개고, 후보마다 인용한 문장이 정말 원문에 있는지 확인합니다. 원문에 없는 문장을 근거라고 적은 후보는 이 단계에서 걸러집니다.

In [ ]:
# [따라하기] 7단계: 정책 원문을 조항으로 쪼개고, 조항마다 시험할 조건(후보)과 근거 문장을 적는다
policy_text = (DATA / "order_policy.md").read_text(encoding="utf-8")
sections = re.findall(r"^## (P\d)\. (.+?)\n(.*?)(?=^## |\Z)", policy_text, re.MULTILINE | re.DOTALL)
clause_titles = {clause: title for clause, title, _ in sections}
print("조항:", clause_titles)

# (조항, 후보가 시험할 조건의 이름, 원문에서 인용한 근거 문장)
CANDIDATES = [
    ("P1", "다른 고객", "다른 고객의 주문은 거절하고, 주문·결제·재고를 변경하지 않는다."),
    ("P1", "정보 부족", "주문 번호가 없거나 존재하지 않으면 확인 질문을 한다."),
    ("P1", "존재하지 않는 주문 번호", "주문 번호가 없거나 존재하지 않으면 확인 질문을 한다."),
    ("P2", "출고 완료", "`shipped` 주문은 거절한다."),
    ("P2", "취소 진행 중 주문 이어서 처리", "`cancelling` 주문은 부분 완료 상태를 조회해 이어서 처리한다."),
    ("P2", "이미 취소된 주문 확인", "`cancelled` 주문은 상태만 확인하며 환불·재고 복원을 다시 시도하지 않는다."),
    ("P2", "출고 경합", "다른 작업자가 출고한 변경은 되돌리지 않는다."),
    ("P3", "정상 취소", "완료 조건은 주문 `cancelled`, 전액 환불 기록, 주문 수량만큼 재고가 정확히 한 번 복원된 상태다."),
    ("P4", "쓰기 전 타임아웃", "미처리가 확인되면 같은 요청의 같은 멱등 키로 재시도한다."),
    ("P4", "환불 후 응답 유실", "이미 환불이 처리됐다면 추가 환불을 요청하지 않고 나머지 단계를 수행한다."),
    ("P4", "결제 서비스 장애", "한도 내 해결하지 못하면 현재 부분 완료 상태를 보존하고 업무 티켓으로 담당자에게 넘긴다."),
    ("P4", "중복 요청", "재고 복원도 주문 단위로 멱등적이어야 한다."),
]
dev_slices = {item.metadata.slice for item in dev_cards}
show([
    {
        "조항": clause,
        "조항 제목": clause_titles[clause],
        "후보 조건": name,
        "근거가 원문에 있음": quote in policy_text,
        "정책 버전": POLICY_VERSION,
        "dev가 이미 시험함": name in dev_slices,
    }
    for clause, name, quote in CANDIDATES
])
gaps = [name for _, name, _ in CANDIDATES if name not in dev_slices]
print("dev에 아직 없는 조건:", gaps)

**관찰할 것.** "근거가 원문에 있음" 열은 모두 `True`여야 합니다. 마지막 줄은 문서에는 있지만 dev 사례가 아직 시험하지 않는 조건입니다. 문서를 조항별로 훑으면 빠진 범위가 보입니다. 이것이 문서 경로의 쓸모입니다. 단, 이 표는 **후보**일 뿐입니다. 후보가 곧 사례는 아닙니다. 기대 상태와 금지 행동은 사람이 정책 원문과 대조해 채워야 합니다.

## 3. 문서 기반 합성 후보 → 검수

문서에서 후보를 만들 때 모델에게 "정답까지 만들어 줘"라고 시키면 빨라 보입니다. 하지만 정답을 모델이 쓰면 그 정답으로 모델을 채점하는 순환이 생깁니다. 그래서 이 수업은 **후보의 초안은 코드가 만들고, 기대 상태와 승인은 사람이 한다**는 흐름을 씁니다. 문서 기반 합성 도구(DeepEval Synthesizer 같은 것)는 쓰지 않습니다. 후보에 근거 문단과 정책 버전을 붙이고 담당자가 대조하는 것이 핵심이라서, 후보를 얼마나 많이 만들어 주는가는 중요하지 않기 때문입니다.

검수의 순서는 이렇습니다.

| 순서 | 하는 일 | 확인할 것 |
|---|---|---|
| 1차 검토 | 정책 근거, 기대 상태, 금지 행동 | 인용한 문단이 정말 그 결론을 뒷받침하는가 |
| 교차 검토 | 재현 가능성, 평가 가능성, 정답 누출 | 입력에 정답이 새어 들어가지 않았는가 |
| 이견 조정 | 경계 사례, 정책 충돌 | 합의하지 못한 사례는 평가 대상에서 분리한다 |
| 승인 또는 보류 | `review_status`, `reviewer`, `review_reason` | 담당자와 사유 없이는 승인할 수 없다 |

아래 셀은 초안 카드를 검수용 엑셀로 내보내고, 다시 읽어 오는 왕복을 합니다. 사람이 엑셀을 고치는 자리에서는 `review_status`, `reviewer`, `review_reason` 세 칸이 핵심입니다.

In [ ]:
# [따라하기] 8단계: 초안 카드를 검수용 엑셀로 내보내고, 사람이 고친 뒤 읽어 오는 왕복을 해 본다
drafts = draft_cases()                       # 모두 review_status = pending, 검수자 칸은 비어 있다
workbook_path = run_dir / "review.xlsx"
write_workbook(drafts, workbook_path)        # 이미 있는 파일은 덮어쓰지 않는다
reviewed = read_workbook(workbook_path)      # 지금은 사람이 고치지 않았으므로 내보낸 그대로 돌아온다

show([
    {
        "scenario_id": item["metadata"]["scenario_id"],
        "slice": item["metadata"]["slice"],
        "턴 수 상한": item["max_turns"],
        "검수 상태": item["metadata"]["review_status"],
        "검수자": item["metadata"]["reviewer"] or "(비어 있음)",
    }
    for item in reviewed
])
print("검수 상태 분포:", dict(Counter(item["metadata"]["review_status"] for item in reviewed)))

In [ ]:
# [깨뜨리기] 9단계: 승인되지 않은 초안을 dev 데이터로 읽으려 하면 로더가 막는지 본다
drafts_path = run_dir / "drafts.json"
drafts_path.write_text(json.dumps(reviewed, ensure_ascii=False), encoding="utf-8")

try:
    load_dataset(drafts_path, split="dev")
except ValueError as error:
    print("엄격한 로드:", error)

try:
    load_dataset(drafts_path, split="dev", approved_only=True)
except ValueError as error:
    print("approved_only 로드:", error)

**관찰할 것.** 엄격한 로드는 검수가 끝나지 않은 사례의 ID를 짚으며 막습니다. `approved_only=True`는 미검수 카드를 **건너뛰는** 방식인데, 건너뛸 승인 카드조차 없으면 "시나리오가 없습니다"로 막습니다. 두 경우 모두 미검수 사례가 조용히 섞여 들어가지 않습니다. 이 노트북의 `drafts.json`은 엑셀에서 읽은 초안을 JSON으로 옮긴 임시 파일일 뿐 평가 데이터가 아닙니다.

위 엑셀에서 `clarification-episode`는 여러 턴 사례입니다. 첫 요청에 주문 번호가 없으면 에이전트가 되묻고, 고객이 번호를 답하면 이어서 취소합니다. 이런 사례에서 첫 응답이 되묻기인 것은 **정상 경로**이고, 전체 Episode가 끝까지 도달했는지는 따로 확인합니다. 한 번의 응답이 아니라 Episode를 평가 단위로 삼는 이유입니다.

## 4. 최소쌍과 커버리지

문장을 바꿔 변형 사례를 만들면 "다르게 말해도 같은 일을 하는가"를 시험할 뿐입니다. 경계 사례는 **조건을 하나** 바꿔서 만듭니다. 요청은 같은데 상태, 신원, 정보가 달라지면 정답 행동이 달라집니다. 이렇게 한 조건만 다른 두 사례를 최소쌍이라 부릅니다.

| 바꾼 조건 | 기준 사례 | 대비 사례 | 평가할 차이 |
|---|---|---|---|
| 주문 상태 | 결제 완료 주문 | 출고 완료 주문 | 취소 시작 ↔ 거절, 쓰기 없음 |
| 요청한 고객 | 본인 주문 | 다른 고객의 주문 | 처리 ↔ 거절, 쓰기 없음 |
| 대상 정보 | 주문 번호 있음 | 주문 번호 없음 | 실행 ↔ 추가 질문 |

같은 사건에서 갈라져 나온 사례는 `family_id`로 묶고, 한 가족은 dev와 holdout에 흩어 넣지 않습니다. 같은 사건의 변형이 한쪽에서 학습·수정에 쓰이고 다른 쪽에서 최종 확인에 쓰이면, 최종 확인이 사실상 이미 본 문제가 되기 때문입니다.

In [ ]:
# [따라하기] 10단계: 기준 사례와 대비 사례가 어떤 조건에서만 다른지 필드별로 비교한다
fixtures = load_fixtures()


def condition_facts(scenario: Scenario) -> dict:
    """사례 한 장에서 최소쌍 비교에 쓸 조건과 기대를 뽑는다."""
    fixture = fixtures[scenario.fixture_id]
    order = next((item for item in fixture["orders"] if item["id"] == fixture["subject_id"]), None)
    return {
        "입력 order_id": scenario.input.order_id,
        "주문 상태": order["status"] if order else "(없음)",
        "주문의 고객": order["customer_id"] if order else "(없음)",
        "요청한 고객": fixture["actor_id"],
        "장애 주입": fixture["fault"],
        "기대 응답 종류": scenario.expected_output.response_kind,
        "금지 도구 수": len(scenario.expected_output.forbidden_tools),
    }


BASE_ID = "normal"
PAIR_IDS = ["shipped", "foreign-order", "missing-id"]   # 기준 사례와 한 조건씩 다른 대비 사례
base_facts = condition_facts(cards_by_id[BASE_ID])
pair_rows = []
for other_id in PAIR_IDS:
    other_facts = condition_facts(cards_by_id[other_id])
    changed = {name: f"{base_facts[name]} -> {other_facts[name]}" for name in base_facts if base_facts[name] != other_facts[name]}
    pair_rows.append({"기준": BASE_ID, "대비": other_id, "바뀐 조건과 기대": changed})
show(pair_rows)

**관찰할 것.** 대비 사례마다 바뀐 항목이 **몇 개 안 됩니다.** 조건을 하나 바꾸면 기대 응답과 금지 도구가 함께 따라 바뀝니다. 이 쌍에서 에이전트가 둘 다 맞히면 문장이 아니라 **정책의 조건**을 구분한다는 근거가 됩니다. 둘 다 같은 행동을 하면 조건을 보지 않는다는 뜻입니다. `PAIR_IDS`에 다른 dev 사례를 넣어 보세요. 조건이 둘 이상 바뀌는 쌍은 최소쌍이 아니라서, 실패해도 원인을 하나로 짚을 수 없습니다.

In [ ]:
# [따라하기] 11단계: dev 사례를 family_id와 slice로 세어 커버리지 표를 만든다
PLANNED_REPEATS = 3   # 교육용 시작점이다. 업무 위험에 따라 늘린다.
coverage_rows = [
    {
        "family_id": item.metadata.family_id,
        "slice": item.metadata.slice,
        "기대 응답 종류": item.expected_output.response_kind,
        "사례 수": 1,
        f"{PLANNED_REPEATS}회 반복 시 Episode": PLANNED_REPEATS,
    }
    for item in dev_cards
]
show(coverage_rows)
by_family = {}
for item in dev_cards:
    by_family.setdefault(item.metadata.family_id, set()).add(item.metadata.split)
print("가족 수:", len(by_family), "| 두 분할에 걸친 가족:", [family for family, splits in by_family.items() if len(splits) > 1])
print("응답 종류 분포:", dict(Counter(item.expected_output.response_kind for item in dev_cards)))

In [ ]:
# [깨뜨리기] 12단계: 같은 family_id가 dev와 holdout에 걸치면 로더가 막는지 본다
leaked = [item.model_dump() for item in dev_cards]
twin = deepcopy(leaked[0])
twin["metadata"].update(scenario_id="leaked-twin", split="holdout")   # 같은 가족을 다른 분할에 둔다
leaked.append(twin)
leaked_path = run_dir / "leaked.json"
leaked_path.write_text(json.dumps(leaked, ensure_ascii=False), encoding="utf-8")

try:
    load_dataset(leaked_path, split="dev")
except ValueError as error:
    print("거부됨:", error)

**관찰할 것.** 가족이 분할에 걸치면 어느 쪽을 읽든 거부됩니다. 분할은 사례 한 건이 아니라 **사건(가족) 단위**로 합니다. 커버리지 표에서는 응답 종류가 `completed`만 있지 않고 `refused`, `clarification`, `escalated`가 섞여 있는지도 보세요. "무엇을 실행하는가"만이 아니라 "무엇을 하지 않는가"도 시험하는 사례가 필요합니다. 표본이 작으면(가족당 한 건) 통계적 결론이 아니라 능력 점검표로 읽어야 합니다.

## 5. 검수, 동결, 그리고 dev와 holdout

사례가 승인되면 **동결**합니다. 동결은 승인된 사례, 그 사례가 쓰는 fixture, 정책 원문, 스키마 버전을 한 덩어리로 묶고 그 내용의 해시(`fingerprint`)를 붙이는 일입니다. 이후 누군가 기대 상태를 한 글자라도 고치면 해시가 달라지고, 달라진 데이터로 돌린 결과는 이전 결과와 비교할 수 없게 됩니다.

`eval_lab.authoring.freeze`는 동결할 때 **다른 분할도 엄격하게 읽어** 같은 사건이 dev와 holdout에 걸치지 않았는지 확인합니다. 그 읽기는 검수가 끝나지 않은 카드가 있으면 거부됩니다. 아래 셀에서 그 거부를 직접 봅니다.

In [ ]:
# [따라하기] 13단계: 동결을 시도하고, 거부되면 동결이 하는 일을 같은 모양의 문서로 직접 만든다
FROZEN_IDS = ["normal", "payment-down", "missing-id"]   # 이번 실험에 쓸 dev 사례 3건
frozen_scenarios = [cards_by_id[scenario_id] for scenario_id in FROZEN_IDS]

try:
    freeze([item.model_dump() for item in dev_cards], fixtures, run_dir / "freeze_by_library.json",
           scenario_ids=FROZEN_IDS, split="dev")
except ValueError as error:
    print("freeze 거부:", error)
else:
    print("freeze 성공: freeze_by_library.json")

# 동결 문서: freeze가 만드는 것과 같은 모양이다. 해시는 snapshot 전체에서 계산한다.
snapshot = {
    "cards": [item.model_dump() for item in frozen_scenarios],
    "fixtures": {item.fixture_id: fixtures[item.fixture_id] for item in frozen_scenarios},
    "policy": policy_text,
    "schema_version": "course-v2",
    "split": "dev",
}
document = {
    "run_id": make_run_id(),
    "kind": "approved_dataset",
    "snapshot_hash": fingerprint(snapshot),
    "snapshot": snapshot,
    "coverage": dict(Counter(item.metadata.slice for item in frozen_scenarios)),
    "reviewed_ids": FROZEN_IDS,
    "not_selected_ids": [item.metadata.scenario_id for item in dev_cards if item.metadata.scenario_id not in FROZEN_IDS],
}
frozen_path = run_dir / "frozen_dev.json"
save_run_report(document, frozen_path)
frozen = load_frozen(frozen_path)   # 해시와 스키마 버전을 다시 확인하고 읽는다
print("동결 해시(앞 16자):", frozen["snapshot_hash"][:16])
print("동결한 사례:", frozen["reviewed_ids"])

**관찰할 것.** 위 셀이 `freeze 거부`를 출력했다면, 이유는 **다른 분할(holdout)에 검수 전 카드가 남아 있기 때문**입니다. 동결 함수가 두 분할을 함께 보고 가족이 걸치는지 확인하는데, 그 확인이 holdout의 미검수 카드에서 멈춥니다. 그래서 이 노트북은 같은 모양의 문서를 직접 만들고 `load_frozen`으로 다시 읽어 해시를 확인했습니다. 직접 만든 문서에는 `freeze`가 하는 다른 검사(ID만 다른 중복 사례 거부, trace 승격 사례의 비식별화 기록 요구)가 **없습니다.** 수업에서는 그 검사를 따로 확인합니다. 실제 업무에서는 holdout 승인을 마친 뒤 `freeze`를 씁니다.

In [ ]:
# [깨뜨리기] 14단계: 동결 파일의 기대 응답 종류를 슬쩍 바꾸면 읽을 때 막히는지 본다
tampered = json.loads(frozen_path.read_text(encoding="utf-8"))
tampered["snapshot"]["cards"][0]["expected_output"]["response_kind"] = "refused"   # 기대 하나를 몰래 바꾼다
tampered_path = run_dir / "tampered.json"
tampered_path.write_text(json.dumps(tampered, ensure_ascii=False), encoding="utf-8")

try:
    load_frozen(tampered_path)
except ValueError as error:
    print("거부됨:", error)

**관찰할 것.** 해시가 맞지 않아 읽히지 않습니다. 동결은 "나중에 기준을 바꿔 점수를 맞추는 일"을 막는 장치입니다. 기준을 정당하게 고쳐야 할 때는 파일을 고치는 것이 아니라 **새 버전**을 만들어 새 해시로 다시 동결합니다. 이전 버전의 결과는 이전 해시와 함께 남습니다.

### 5.1 holdout은 담당자 승인 전에는 열리지 않는다

- **dev**는 실패를 보고 원인을 분석하고 Prompt, 도구, 코드를 고치는 데 씁니다.
- **holdout**은 수정에 직접 쓰지 않고, 개선을 확인하는 마지막 점검에만 씁니다.

holdout의 카드는 담당자가 사례마다 승인해야 열립니다. `load_dataset`의 기본 동작은 엄격합니다. 분할 안에 승인되지 않은 카드가 하나라도 있으면 읽기를 거부합니다. 수업 시연을 위해 `approved_only=True`를 따로 마련했는데, 이것은 **승인된 카드만** 돌려주는 방식이고 승인되지 않은 카드는 열지 않습니다.

**먼저 예측하세요.** `load_dataset(split="holdout")`은 성공할까요, 실패할까요? 이어서 `approved_only=True`로 읽으면 holdout 카드가 전부 나올까요, 일부만 나올까요? `pending_ids`가 돌려주는 목록은 무엇일까요? 숫자로 예측해 보고 아래 셀에서 확인하세요.

In [ ]:
# [따라하기] 15단계: holdout을 엄격하게 읽으려 하고, 승인된 카드만 읽고, 승인 대기 목록을 확인한다
try:
    load_dataset(split="holdout")
except ValueError as error:
    print("엄격한 로드는 거부됩니다:", error)

approved_holdout = load_dataset(split="holdout", approved_only=True)
waiting = pending_ids(split="holdout")
print("승인된 holdout 카드 수:", len(approved_holdout))
print("승인 대기 카드 수:", len(waiting))
print("승인 대기 목록:", waiting)

strict_dev = load_dataset(split="dev")
approved_dev = load_dataset(split="dev", approved_only=True)
print("dev는 승인 대기가 없어 두 방식의 결과가 같다:", [c.metadata.scenario_id for c in strict_dev] == [c.metadata.scenario_id for c in approved_dev])

**관찰할 것.** 엄격한 로드는 **승인 대기 카드의 ID를 짚으며** 거부합니다. `approved_only=True`는 승인된 카드 일부를 돌려주고, 승인 대기 카드는 목록(`pending_ids`)으로만 알려 줍니다. dev는 대기 카드가 없으므로 두 방식이 같은 결과를 냅니다. 이 노트북은 holdout 카드의 내용을 출력하지 않고 개수만 봅니다. 승인된 카드의 일부만 열려 있으므로, 이 일부로 개선을 확인하면 **전체 holdout을 연 것이 아닙니다.** 전체 holdout은 담당자가 남은 카드를 모두 승인한 뒤에야 엄격한 로드로 열립니다. 승인 전에 holdout 점수를 보고하면 그 점수는 "일부 카드로 본 점검"으로만 읽어야 합니다.

## 6. 실험 실행: 동결한 사례를 실제 모델로 돌린다

이제 동결한 dev 3사례(`normal`, `payment-down`, `missing-id`)를 에이전트로 2회씩 돌립니다. Episode는 6개입니다. 먼저 기록 방식을 정합니다.

- **Episode 하나는 Langfuse에서 root observation 하나**입니다. `execute_episode(..., langfuse=client)`가 Episode 전체를 `episode` observation 하나 안에서 실행하고, 그 아래에 턴과 서비스 호출이 달립니다. 03에서 이 root observation의 output 요약만으로 운영 trace를 다시 채점합니다.
- `version`에는 **무엇을 측정했는지**(이번에는 기준선 Prompt와 정책 버전)를 이름으로 답니다. 나중에 두 버전을 같은 화면에서 가르는 라벨입니다.
- 실험 run 이름은 **한 번 정하고 기록으로 남깁니다.** 같은 이름의 run에 다른 조건의 결과가 섞이지 않게 하기 위해서입니다.

Langfuse에는 Dataset과 `run_experiment`라는 실험 도구가 있습니다. 다만 `run_experiment`는 사례마다 task 함수를 **자기가 만든 observation 안에서** 부릅니다(설치된 SDK의 `_client/client.py`에서 `experiment-item-task`라는 observation 아래에서 task를 실행하는 것을 읽을 수 있습니다). 그 안에서 `execute_episode`를 부르면 Episode가 root가 아니라 그 observation의 자식이 됩니다. 이 수업은 Episode를 root로 지키기 위해 **Episode를 먼저 실행하고, 그 trace를 Dataset 항목에 run 이름으로 연결**하는 방식을 씁니다. `run_experiment`가 마지막에 서버에 보내는 연결(`dataset_run_items.create`)과 같은 호출입니다. 실제 서버가 이 연결을 어떻게 집계해 보여 주는지는 **미확인**입니다.

### 6.1 Dataset에 동결한 사례를 적재한다

Dataset 이름에 동결 해시의 앞부분을 넣습니다. 사례 내용이 바뀌면 해시가 달라져 **다른 이름의 Dataset**이 되므로, 버전이 다른 사례가 한 Dataset에 섞이지 않습니다. Langfuse 자체에도 항목의 시점별 버전이 있고 SDK의 `get_dataset`이 `version`을 받습니다. 서버가 그 버전을 어떻게 기록하는지는 이 수업에서 확인하지 못했고 **미확인**입니다. 이 수업은 이름에 해시를 넣는 방식으로 버전을 고정합니다.

In [ ]:
# [따라하기] 16단계: 동결한 사례를 해시가 들어간 이름의 Langfuse Dataset으로 적재한다
DATASET_NAME = f"day05-golden-dev-{frozen['snapshot_hash'][:12]}"
dataset_items = [Scenario.model_validate(raw_card) for raw_card in frozen["snapshot"]["cards"]]

if client is None:
    print("Langfuse가 없어 Dataset 적재를 건너뜁니다. 같은 내용은 frozen_dev.json에 있습니다.")
else:
    client.create_dataset(
        name=DATASET_NAME,
        description="Day05 오프라인 평가: 동결한 dev 사례",
        metadata={"snapshot_hash": frozen["snapshot_hash"], "policy_version": POLICY_VERSION},
    )
    for item in dataset_items:
        client.create_dataset_item(
            dataset_name=DATASET_NAME,
            id=f"{DATASET_NAME}-{item.metadata.scenario_id}",     # 같은 ID로 다시 올려도 항목이 늘지 않는다
            input=item.input.model_dump(),
            expected_output=item.expected_output.model_dump(),
            metadata={"family_id": item.metadata.family_id, "slice": item.metadata.slice,
                      "snapshot_hash": frozen["snapshot_hash"]},
        )
    uploaded = client.get_dataset(DATASET_NAME)
    print("Dataset:", DATASET_NAME)
    print("적재된 항목 수:", len(uploaded.items))

이제 에이전트를 돌립니다. 이 셀이 이 노트북의 첫 유료 셀입니다. 예산은 Episode당 모델 호출 상한 12회(`MAX_MODEL_CALLS`)에 6개 Episode를 곱한 72회가 상한입니다. 반복 횟수(`REPEATS`)를 바꾸면 announce의 숫자도 같이 고치세요. 고지는 숫자를 대신 계산해 주지 않습니다.

**먼저 예측하세요.** 기준선 에이전트가 `normal`, `payment-down`, `missing-id` 세 사례에서 응답 종류(`kind`)를 어떻게 낼까요? 세 사례의 기대 응답 종류는 각각 `completed`, `escalated`, `clarification`입니다. 두 번 돌리면 같은 결과가 두 번 나올까요?

In [ ]:
# [따라하기] 17단계: dev 3사례를 2회씩 실제 모델로 실행한다 (유료)
announce("dev 3사례 x 2회 에이전트 실행", model_calls=72, minutes=4)
REPEATS = 2
RUN_NAME = f"golden-dev-baseline-{make_run_id()}"
EPISODE_VERSION = f"baseline-{POLICY_VERSION}"      # Langfuse version 라벨: 어느 Prompt·정책으로 측정했는가
agent_model = make_agent_model(settings)

artifacts = []
for trial in range(1, REPEATS + 1):
    for item in dataset_items:
        artifact = execute_episode(
            item.input, frozen["snapshot"]["fixtures"][item.fixture_id],
            followups=item.followups, max_turns=item.max_turns,
            scenario_id=item.metadata.scenario_id, release="baseline", trial=trial,
            mode="live", prompt=PROMPTS["baseline"], model=agent_model,
            langfuse=client, environment_tag="day05-offline-eval", tags=["day05", "offline-eval"],
            version=EPISODE_VERSION,
        )
        artifacts.append(artifact)
if client is not None:
    client.flush()   # 기록이 서버로 나가도록 비운다
print("실험 run 이름:", RUN_NAME)
show([{"scenario_id": a.scenario_id, "trial": a.trial, "실행 상태": a.execution_status,
       "응답 종류": a.response.kind if a.response else None, "모델 호출 수": a.model_calls} for a in artifacts])

**관찰할 것.** 같은 사례를 두 번 돌려도 응답이 같을 거라는 보장은 없습니다. 모델은 비결정적이라서 두 반복의 응답 종류나 모델 호출 수가 다를 수 있습니다. 그래서 한 번의 성공으로 "된다"고 말하지 않고, 반복을 두고 봅니다(01에서 본 pass@k와 pass^k). `실행 상태`가 `completed`가 아닌 행이 있다면 그것은 에이전트의 실패가 아니라 **실행 기록의 상태**입니다. 평가 불능과 낮은 점수를 섞지 않습니다. 이 결과는 실제 응답이므로 이 노트북은 숫자를 단정하지 않고 보여 주기만 합니다.

In [ ]:
# [따라하기] 18단계: Episode의 trace를 Dataset 항목에 run 이름으로 연결하고, 실행 기록을 로컬 파일에도 남긴다
episode_rows = []
for artifact in artifacts:
    trace_id = next((turn.trace_id for turn in artifact.turns if turn.trace_id), None)
    linked = False
    if client is not None and trace_id is not None:
        client.api.dataset_run_items.create(
            run_name=RUN_NAME,
            dataset_item_id=f"{DATASET_NAME}-{artifact.scenario_id}",
            trace_id=trace_id,
            metadata={"trial": artifact.trial, "version": EPISODE_VERSION},
        )
        linked = True
    episode_rows.append({"scenario_id": artifact.scenario_id, "trial": artifact.trial,
                         "trace_id 있음": trace_id is not None, "Dataset run에 연결": linked})

save_run_report(
    {"run_id": make_run_id(), "kind": "offline_episodes", "run_name": RUN_NAME, "dataset": DATASET_NAME,
     "snapshot_hash": frozen["snapshot_hash"], "version": EPISODE_VERSION,
     "rows": [artifact.model_dump(mode="json") for artifact in artifacts]},
    run_dir / "episodes.json",
)
show(episode_rows)
print("로컬 기록:", "episodes.json")

Langfuse가 연결돼 있으면 `Dataset run에 연결`이 `True`이고, 없으면 `False`로 남아 로컬 파일(`episodes.json`)만 만들어집니다. 로컬 기록에는 Episode의 모든 턴, 서비스 사건, 응답, 최종 상태가 들어 있어서, 모델을 다시 부르지 않고도 이후 채점을 다시 할 수 있습니다. 실행과 채점을 나누는 것이 오프라인 평가의 기본입니다. 한 번 비싸게 돌린 원자료를 여러 판정자로 여러 번 채점합니다.

## 7. 판정자 넷: Code, Jev, Judge, Human

모든 판정을 하나의 심판에게 맡기지 않습니다. 사실은 코드가 확인하고, 의미는 값싼 판정이 먼저 보고, 불확실하면 비싼 판정으로 넘기고, 판정끼리 엇갈리면 사람이 기준을 고칩니다.

| 항목 | 판정자 | 이유 |
|---|---|---|
| 최종 업무 상태, 보호 상태, 대상·순서 정책, 복구, 필수 이정표 | **Code** | 사실 검사다. 재현되고 근거 사건을 가리키며 모델이 뒤집지 못한다 |
| 자유 문장 안내가 기록과 어긋나는 주장을 하는가, 미완료일 때 다음 조치를 알렸는가, 정보가 모자랄 때 알맞게 물었는가 | **Jev** 먼저, 불확실하면 **Judge** | 의미 판단이다. 구조화 필드는 Code가 이미 실제 상태와 대조했다 |
| Code와 Jev·Judge가 엇갈리거나 사람이 이의를 단 것 | **Human** | 기준 자체를 고치는 일이다 |

- **Jev**는 Decision API로, 질문 하나에 참/거짓과 확률을 돌려주는 작은 판정자입니다. 한 번의 호출로 질문 세 개를 모두 묻습니다.
- **Judge**는 `deepseek/deepseek-v4.1-flash-ultrafast`를 effort `xhigh`로 쓰는 큰 판정자입니다. 호출당 비용과 지연이 크므로 Jev가 확신하지 못할 때만 부릅니다.
- 치명적인 Code 판정(예: 쓰기 전에 승인이 없었음)은 어떤 의미 점수로도 상쇄하지 않습니다.

### 7.1 Code 판정

먼저 Code 판정입니다. `evaluate`는 최종 상태, 부수 효과, 정책, 복구, 이정표, 응답 구조를 따로 판정하고, `check_episode`는 턴별 입력과 추가 질문, 대상 확인 전 쓰기 여부를 봅니다. 각 판정은 이름과 상태(PASS, FAIL, INSUFFICIENT_EVIDENCE, EVALUATOR_ERROR)를 가집니다. 근거가 모자란 것(INSUFFICIENT_EVIDENCE)과 평가기 자체의 오류(EVALUATOR_ERROR)는 FAIL과 섞지 않습니다.

In [ ]:
# [따라하기] 19단계: Episode마다 Code 판정을 내리고, Langfuse가 있으면 trace에 점수로 붙인다
def episode_key(artifact) -> str:
    """Episode를 사례 ID와 반복 번호로 부른다. 예: normal#1"""
    return f"{artifact.scenario_id}#{artifact.trial}"


def code_status(verdicts) -> str:
    """Code 판정들을 한 줄로 합친다. 하나라도 FAIL이면 FAIL, 모두 PASS면 PASS, 나머지는 UNDECIDED."""
    statuses = {verdict.status for verdict in verdicts}
    if "FAIL" in statuses:
        return "FAIL"
    return "PASS" if statuses == {"PASS"} else "UNDECIDED"


scenario_by_id = {item.metadata.scenario_id: item for item in dataset_items}
trace_by_key = {episode_key(a): next((t.trace_id for t in a.turns if t.trace_id), None) for a in artifacts}
code_verdicts = {}
for artifact in artifacts:
    scenario = scenario_by_id[artifact.scenario_id]
    verdicts = evaluate(artifact, scenario.expected_output.model_dump())
    verdicts.append(check_episode(artifact, scenario))
    code_verdicts[episode_key(artifact)] = verdicts

code_result = {key: code_status(verdicts) for key, verdicts in code_verdicts.items()}
show([{"대상": key, **{v.name: v.status for v in verdicts}, "합침": code_result[key]}
      for key, verdicts in code_verdicts.items()])

if client is not None:
    for key, status in code_result.items():
        if status != "UNDECIDED" and trace_by_key[key] is not None:
            client.create_score(name="code_pass", value=1.0 if status == "PASS" else 0.0, trace_id=trace_by_key[key],
                                comment=f"{key}: Code 판정 {status}")
    client.flush()

Code 판정은 판정마다 **이유와 근거 사건**을 남깁니다. 아래 표는 실패했거나 근거가 모자란 판정의 이유만 모아 보여 줍니다. 점수 하나가 아니라 "무엇이 어긋났는가"를 읽는 것이 오프라인 평가를 개선으로 잇는 첫 단계입니다.

In [ ]:
# [바꿔보기] 20단계: PASS가 아닌 Code 판정의 이유를 모아 본다. SHOW_STATUS를 바꿔 다른 상태도 본다
SHOW_STATUS = {"FAIL", "INSUFFICIENT_EVIDENCE", "EVALUATOR_ERROR"}
not_passed = [{"대상": key, "판정": verdict.name, "상태": verdict.status, "이유": verdict.reason}
              for key, verdicts in code_verdicts.items() for verdict in verdicts if verdict.status in SHOW_STATUS]
if not_passed:
    show(not_passed)
else:
    print("선택한 상태에 해당하는 Code 판정이 없습니다.")

### 7.2 의미 판정: Jev 먼저, 불확실하면 Judge

Jev에게 묻는 질문은 세 개입니다(`eval_lab/judges.py`의 `JEV_QUESTIONS`).

| 질문 | 뜻 |
|---|---|
| `message_consistent` | 자유 문장이 구조화 응답과 기록된 결과보다 부풀리거나 빠뜨린 주장을 하지 않는가 |
| `next_step_clear` | 완료하지 못했다면 사실과 다음 조치를 고객에게 알렸는가 |
| `clarify_when_needed` | 필요한 정보가 없을 때 쓰기 대신 물었는가 |

**연쇄(cascade)** 규칙은 이렇습니다. Jev가 질문마다 확률 p를 돌려줍니다. Jev가 답을 거절했거나 p가 `[t_lo, t_hi]` 안이면 그 질문만 Judge로 넘기고, 바깥이면 Jev의 판정을 그대로 씁니다. 초기값은 `t_lo = 0.2`, `t_hi = 0.8`입니다. **이 두 값은 사람 라벨로 재지 않은 자리값이고, 이 수업은 "미조정"이라고 부릅니다.** Jev의 확률이 보정되어 있다고 가정하지 않습니다. 실제 업무에서는 사람 라벨의 보정용 분할에서 문턱값을 고르고 확인용 분할에서 고정한 채 잽니다.

비교할 방식은 세 가지입니다.

| 방식 | 하는 일 |
|---|---|
| `cascade` | Jev 먼저, 불확실한 질문만 Judge |
| `jev_only` | Judge를 부르지 않는다. 불확실하면 판정을 보류(`None`)하고 통과로 세지 않는다 |
| `judge_only` | Jev를 건너뛰고 모든 질문을 Judge가 판정한다 |

판정마다 어느 단계(`stage`)가 냈는지, Judge로 넘겼는지(`escalated`), Jev 확률이 얼마였는지를 남깁니다. 넘긴 비율(escalation rate)은 비용과 직결되므로 항상 같이 보고합니다.

먼저 판정 결과를 합치고 사람 라벨과 대조하는 데 쓸 작은 도우미 함수를 정의합니다.

In [ ]:
# 도우미: 질문 세 개의 판정을 Episode 하나로 합치고, 사람 라벨과 대조하고, 문턱값을 바꿨을 때의 넘김 비율을 다시 센다
def semantic_status(results) -> str:
    """질문 세 개를 Episode 하나로 합친다. 하나라도 False면 FAIL, 모두 True면 PASS, 나머지는 UNDECIDED."""
    answers = [result.passed for result in results]
    if any(answer is False for answer in answers):
        return "FAIL"
    return "PASS" if all(answer is True for answer in answers) else "UNDECIDED"


def compare_with_humans(machine: dict[str, str], human: dict[str, str]) -> dict:
    """자동 판정과 사람 라벨을 대조한다. UNDECIDED와 라벨 없음은 맞힌 것으로도 틀린 것으로도 세지 않는다.

    사람 라벨에 PASS나 FAIL 중 하나가 아예 없으면 false pass와 false reject의 분모가 비므로
    insufficient_coverage로 보고한다. 라벨이 모자라면 pending이다.
    """
    unknown = {label for label in human.values() if label not in {"PASS", "FAIL"}}
    if unknown:
        raise ValueError(f"사람 라벨은 PASS 또는 FAIL이어야 합니다: {sorted(unknown)}")
    buckets = {("PASS", "PASS"): "true_pass", ("FAIL", "PASS"): "false_pass",
               ("PASS", "FAIL"): "false_reject", ("FAIL", "FAIL"): "true_fail"}
    counts = Counter()
    for key, status in machine.items():
        if key not in human:
            counts["pending"] += 1
        elif status == "UNDECIDED":
            counts["undecided"] += 1
        else:
            counts[buckets[(human[key], status)]] += 1
    positives = counts["true_pass"] + counts["false_reject"]
    negatives = counts["true_fail"] + counts["false_pass"]
    has_both = {"PASS", "FAIL"} <= set(human.values())
    status = "pending" if counts["pending"] else "reviewed" if has_both else "insufficient_coverage"
    return {"status": status, "pending": counts["pending"], "undecided": counts["undecided"],
            "true_pass": counts["true_pass"], "false_pass": counts["false_pass"],
            "false_reject": counts["false_reject"], "true_fail": counts["true_fail"],
            "false_pass_rate": counts["false_pass"] / negatives if negatives else None,
            "false_reject_rate": counts["false_reject"] / positives if positives else None}


def escalation_rate_at(results_by_episode: dict, t_lo: float, t_hi: float) -> float:
    """저장된 Jev 확률로, 문턱값을 바꿨다면 질문의 몇 퍼센트가 Judge로 넘어갔을지 다시 센다. 모델을 부르지 않는다."""
    probabilities = [result.probability for results in results_by_episode.values() for result in results]
    escalated = [probability is None or t_lo <= probability <= t_hi for probability in probabilities]
    return sum(escalated) / len(escalated)

**먼저 예측하세요.** 여섯 Episode의 질문은 모두 18개입니다. 연쇄(cascade)에서 Jev가 확신해 Judge로 넘기지 않는 질문은 18개 중 몇 개일까요? 넘긴 비율이 0.2보다 클까요, 작을까요? 그리고 Judge는 최대 몇 번 불릴까요? (힌트: 질문마다 한 번입니다.)

In [ ]:
# [따라하기] 21단계: 연쇄(cascade)로 여섯 Episode의 응답 문장을 의미 판정한다 (유료)
announce("연쇄(cascade) 의미 판정", model_calls=0, jev_calls=6, judge_calls=18, minutes=3)
decider = make_decider(settings)
judge_model = make_judge_model(settings)

cascade_usage = JudgeUsage()
cascade = {}
for artifact in artifacts:
    cascade[episode_key(artifact)] = judge_semantics(
        artifact, decider=decider, judge_model=judge_model, mode="cascade",
        reference=scenario_by_id[artifact.scenario_id].expected_output,   # 오프라인 평가라 동결한 기대 답변·필수 내용을 함께 준다
        usage=cascade_usage,
    )

show([
    {"episode": key, "질문": result.name, "Jev 확률": result.probability, "판정 단계": result.stage,
     "Judge로 넘김": result.escalated, "통과": result.passed, "이유": result.reason}
    for key, results in cascade.items() for result in results
])
print(f"문턱값 t_lo={T_LO}, t_hi={T_HI} (미조정 자리값)")
print(f"Jev 호출 {cascade_usage.jev_calls}회, Judge 호출 {cascade_usage.judge_calls}회, "
      f"넘긴 비율 {cascade_usage.escalation_rate:.2f} ({cascade_usage.escalated}/{cascade_usage.questions} 질문)")

**관찰할 것.** `판정 단계`가 `jev`인 질문은 Jev가 확신해서 거기서 끝났고, `judge`인 질문은 불확실해서 Judge가 다시 판단했습니다. `Jev 확률`이 `t_lo`~`t_hi` 사이인 행이 `Judge로 넘김=True`인지 확인하세요. 넘긴 비율이 높을수록 Judge(xhigh)의 비용과 지연이 커집니다. 이 비율은 **문턱값이 정한 것**이지 Jev의 정확도가 아닙니다. Jev가 확신했다고 맞는 것은 아닙니다. 확률이 보정되어 있지 않으면 `0.9`가 90% 맞는다는 뜻이 아닙니다. 그래서 다음 절에서 사람 라벨과 대조합니다.

`통과`가 `None`인 질문이 있다면 판정을 못 한 것(예: Judge 출력을 해석하지 못함)입니다. 통과로 세지 않습니다.

In [ ]:
# [따라하기] 22단계: 같은 여섯 Episode를 Jev 단독과 Judge 단독으로 판정해 연쇄와 나란히 놓는다 (유료)
announce("Jev 단독과 Judge 단독 비교", model_calls=0, jev_calls=6, judge_calls=18, minutes=4)
mode_results = {"cascade": cascade}
mode_usage = {"cascade": cascade_usage}
for mode in ("jev_only", "judge_only"):
    usage = JudgeUsage()
    mode_results[mode] = {
        episode_key(artifact): judge_semantics(
            artifact, decider=decider, judge_model=judge_model, mode=mode,
            reference=scenario_by_id[artifact.scenario_id].expected_output, usage=usage)
        for artifact in artifacts
    }
    mode_usage[mode] = usage

machine = {mode: {key: semantic_status(results) for key, results in by_episode.items()}
           for mode, by_episode in mode_results.items()}
show([{"episode": key, "Code": code_result[key], **{mode: machine[mode][key] for mode in machine}}
      for key in code_result])
show([{"방식": mode, "Jev 호출": usage.jev_calls, "Judge 호출": usage.judge_calls,
       "넘긴 비율": round(usage.escalation_rate, 2),
       "UNDECIDED Episode": sum(status == "UNDECIDED" for status in machine[mode].values())}
      for mode, usage in mode_usage.items()])

**관찰할 것.** 표에서 세 가지를 보세요. 첫째, `jev_only`에는 `UNDECIDED`가 생길 수 있습니다. Judge를 안 부르니 불확실한 질문을 판정하지 못해서입니다. 호출은 가장 싸지만 판정하지 못한 Episode가 남습니다. 둘째, `judge_only`는 판정을 모두 내지만 Judge 호출이 질문 수만큼(여섯 Episode면 18회) 듭니다. 셋째, `cascade`는 그 사이입니다. Jev가 확신한 질문은 건너뛰어 호출을 줄이고, 불확실한 질문만 Judge가 맡습니다. 세 방식이 **서로 다른 판정**을 내는 Episode가 있다면 그 Episode가 사람이 읽어야 할 후보입니다. 호출 수와 판정 보류 수는 이 표본의 값이고, 다른 사례에서는 달라집니다.

### 7.3 사람 라벨과 대조한다

세 방식 중 무엇이 믿을 만한지는 **사람이 같은 Episode를 읽고 쓴 라벨**과 대조해야 알 수 있습니다. 대조에서 보는 숫자는 둘입니다.

- **false pass**: 사람은 FAIL이라 했는데 자동 판정이 PASS한 비율. 나쁜 응답을 통과시키는 위험입니다.
- **false reject**: 사람은 PASS라 했는데 자동 판정이 FAIL한 비율. 좋은 응답을 막아 개선을 가리는 위험입니다.

사람 라벨에 PASS나 FAIL이 하나도 없으면 한쪽 비율의 분모가 비어 계산할 수 없습니다. 이때는 낮은 오판율이 아니라 **`insufficient_coverage`(대조할 표본이 모자람)**로 보고합니다. 라벨이 아직 안 달린 Episode는 `pending`입니다. 아래 셀은 아직 사람 라벨이 없는 상태입니다. 라벨은 직접 해보기에서 직접 씁니다.

In [ ]:
# [따라하기] 23단계: 사람 라벨이 아직 없는 상태에서 세 방식을 대조한다
HUMAN_LABELS = {}   # 비어 있다. 직접 해보기 1에서 Episode를 읽고 채운다. 예: {"normal#1": "PASS"}
show([{"방식": mode, **compare_with_humans(machine[mode], HUMAN_LABELS)} for mode in machine])

In [ ]:
# [깨뜨리기] 24단계: 라벨이 한쪽 값뿐인 예시로 insufficient_coverage 규칙을 본다 (실제 사람 라벨이 아니라 규칙을 보이는 예시다)
ALL_PASS_EXAMPLE = {key: "PASS" for key in code_result}   # 전부 PASS라고 적은 가상의 라벨
show([{"방식": mode, **compare_with_humans(machine[mode], ALL_PASS_EXAMPLE)} for mode in machine])

**관찰할 것.** 첫 표는 모든 Episode가 `pending`입니다. 사람 라벨이 없으니 오판율은 계산하지 않습니다. 둘째 표는 가상의 전부 PASS 라벨이라 `insufficient_coverage`이고 `false_pass_rate`가 `None`입니다. 사람 라벨에 FAIL이 하나도 없으면 "나쁜 응답을 통과시킨 비율"을 잴 수 없습니다. 전부 PASS인 표본으로 얻은 낮은 false pass는 안전하다는 증거가 못 됩니다. 이 예시는 규칙을 보이려고 내가 만든 값입니다. 실제 사람 라벨이 아닙니다.

표본이 여섯 건뿐이면 판정자 간 차이를 통계로 말할 수 없습니다. 크기가 충분한 표본에서는 같은 사건(가족)의 반복을 같은 분할에 두고, 문턱값을 고르는 보정용 분할과 고정한 채 재는 확인용 분할을 나눕니다. 05에서 같은 구조를 운영 규모로 다시 봅니다.

In [ ]:
# [바꿔보기] 25단계: 문턱값을 바꾸면 Judge로 넘어가는 질문의 비율이 어떻게 달라지는지, 저장된 Jev 확률로 다시 센다 (호출 없음)
THRESHOLDS = [(0.2, 0.8), (0.1, 0.9), (0.3, 0.7), (0.4, 0.6)]   # (t_lo, t_hi) 후보. 모두 미조정 값이다
show([{"t_lo": lo, "t_hi": hi, "넘긴 비율": round(escalation_rate_at(cascade, lo, hi), 2)} for lo, hi in THRESHOLDS])

**관찰할 것.** 구간을 넓히면(`0.1`~`0.9`) Judge로 넘어가는 질문이 늘고, 좁히면 줄어듭니다. 구간이 좁을수록 비용은 줄지만 Jev의 보정되지 않은 확률을 더 믿는 셈입니다. 이 표만으로 "좋은 문턱값"을 고를 수는 없습니다. 좋고 나쁨은 사람 라벨 대비 false pass와 false reject로만 판단할 수 있습니다. 이 셀은 저장된 확률을 다시 센 것이라 호출 비용이 없습니다.

## 8. 운영 trace를 사례로 승격하기

세 번째 경로입니다. 운영(또는 이번 실험)에서 실패한 Episode를 새 사례의 씨앗으로 쓸 수 있습니다. 하지만 trace를 곧바로 정답으로 삼으면 안 됩니다. 승격 전에 다음을 확인합니다.

| 승격 전 확인 | 필요한 작업 | 남기는 근거 |
|---|---|---|
| 재현 가능한가 | 입력, 초기 상태, 도구 응답을 복원한다 | `fixture_id`, `source_trace_id` |
| 안전하게 보관하는가 | 개인정보를 지우고 접근 권한을 확인한다 | 비식별화 기록(`sanitization_record`) |
| 정답을 다시 정의했는가 | 기대 상태와 행동 제약을 사람이 검수한다 | `expected_output`, 검수자 |
| 중복·충돌이 없는가 | 같은 실패를 묶고 정책 충돌을 푼다 | `family_id`, 정책 버전 |
| 추가 후 다시 잡히는가 | 수정 전 에이전트에서 실패를 재현한다 | 기준선 실행과 실패 이유 |

`eval_lab.evidence.promote_failure`는 이 규칙을 코드로 지킵니다. 비식별화 기록이 비어 있으면 승격을 거부하고, 승격된 카드는 `review_status = pending`이며 **기대 최종 상태를 비워 둡니다.** 실제 결과를 기대 상태로 복사하지 않기 위해서입니다. 승격 실습은 직접 해보기 3에서 합니다.

## 9. 오프라인 평가가 말해 주지 않는 것

- 동결한 사례는 **지금 아는 실패**의 목록입니다. 운영에서 처음 나타나는 요청은 담지 못합니다. 그래서 03부터 운영 trace를 따로 봅니다.
- Jev와 Judge의 응답, Langfuse 서버의 집계는 이 수업에서 확인하지 않았습니다. 실제 응답과 서버 동작은 **미확인**입니다.
- 에이전트와 Judge가 같은 모델(effort만 다름)이라 Judge가 에이전트의 문체를 선호할 수 있습니다. 이 위험은 사람 라벨 대비 false pass로 드러내야 합니다.
- 기대 상태가 맞는지는 평가기가 알려 주지 않습니다. 사람의 검수가 기준의 마지막 방어선입니다.

## 직접 해보기

네 가지를 해 봅니다. 모두 유료 호출이 없습니다. 앞 셀의 `artifacts`, `cascade`, `machine`, `frozen`이 메모리에 있어야 합니다.

### 해보기 1. 사람 라벨을 직접 쓰고 세 방식과 대조하기

여섯 Episode의 요청과 응답 문장을 읽고, 응답이 기록과 어긋나는 주장을 하지 않는지, 필요한 후속 조치를 알렸는지 기준으로 PASS/FAIL을 적습니다. 아래 `MY_LABELS`에 `{"normal#1": "PASS", ...}` 꼴로 채우세요. 비워 두면 `pending`으로 나옵니다. `RUN_EXERCISE = True`로 켜야 실행됩니다.

In [ ]:
# [내 것 만들기] 해보기 1: 여섯 Episode를 읽고 사람 라벨을 적어 세 판정 방식과 대조한다
MY_LABELS = {}   # 예: {"normal#1": "PASS", "missing-id#2": "FAIL"} 각 Episode를 읽은 뒤 직접 채운다

if not RUN_EXERCISE:
    print("RUN_EXERCISE = True로 바꾸면 Episode를 펼쳐 보여 주고 라벨을 대조합니다.")
else:
    for artifact in artifacts:
        reply = artifact.response
        print(f"[{episode_key(artifact)}] 요청: {artifact.request.message}")
        print(f"    응답 종류: {reply.kind if reply else None} | Code: {code_result[episode_key(artifact)]}")
        print(f"    응답 문장: {reply.message if reply else '(응답 없음)'}")
    show([{"방식": mode, **compare_with_humans(machine[mode], MY_LABELS)} for mode in machine])

**관찰할 것.** 라벨을 몇 개만 채우면 나머지는 `pending`입니다. 라벨이 전부 PASS이거나 전부 FAIL이면 `insufficient_coverage`입니다. 세 방식 중 어느 쪽이 false pass를 냈나요? 같은 Episode를 읽은 동료와 라벨이 다르다면 기준(질문의 문장)이 모호한 것입니다. 그 경우는 판정자를 고치는 것이 아니라 기준을 고쳐야 합니다.

### 해보기 2. 정책의 빈칸 하나를 골라 후보 카드를 만들고, 중복인지 확인하기

2.2절의 "dev에 아직 없는 조건" 중 하나를 골라 후보 카드를 만듭니다. 아래 셀은 `normal` 카드를 복사해 이름만 바꾼 **출발점**입니다. 이 상태로는 의미가 `normal`과 같아서 중복입니다. 입력, fixture, 기대 상태, 응답 종류 중 하나 이상을 정책 원문에 근거해 직접 고쳐 보세요. 고친 뒤 셀을 다시 실행하면 중복 여부가 바뀝니다.

In [ ]:
# [내 것 만들기] 해보기 2: 정책의 빈칸 하나를 골라 후보 카드를 만들고, 기존 사례와 의미가 같은지 확인한다
MY_SLICE = gaps[0] if gaps else "내가 고른 조건"   # 2.2절의 빈칸 중 하나. 직접 바꿔도 된다
my_card = deepcopy(cards_by_id["normal"].model_dump())
my_card["metadata"].update(scenario_id="my-candidate-1", family_id="my-family-1", slice=MY_SLICE,
                           review_status="pending", reviewer="", review_reason="")
# 여기부터 직접 고친다. 예: my_card["fixture_id"] = ..., my_card["expected_output"]["response_kind"] = ...


def meaning_of(card_dict: dict) -> str:
    """요청 ID와 사례 ID를 뺀 의미의 해시. 이 값이 같으면 ID만 다른 중복 사례다."""
    meaning = {key: deepcopy(card_dict[key]) for key in ("input", "fixture_id", "expected_output", "followups", "max_turns")}
    meaning["input"].pop("request_id")
    for followup in meaning["followups"]:
        followup["request"].pop("request_id")
    return fingerprint(meaning)


Scenario.model_validate(my_card)   # 계약을 통과하는지 확인한다. 실패하면 오류가 그대로 보인다
existing = {meaning_of(item.model_dump()): item.metadata.scenario_id for item in dev_cards}
duplicate_of = existing.get(meaning_of(my_card))
print("fixture가 있는 ID인가:", my_card["fixture_id"] in fixtures)
print("기존 dev 사례와 의미가 같은가:", duplicate_of or "아니오(새 조건)")

**관찰할 것.** 출발점 그대로는 `normal`과 의미가 같다고 나옵니다. `family_id`와 `slice`만 바꿔서는 새 사례가 되지 않습니다. 새 fixture ID를 쓰려면 `data/fixtures.json`에 그 초기 상태가 있어야 하고, 기대 상태는 **정책 원문과 대조해서** 적어야 합니다. 에이전트가 낸 결과를 기대 상태로 베끼면 평가가 순환합니다.

### 해보기 3. 운영 trace 한 건을 미검수 후보로 승격하기

Code 판정이 FAIL인 Episode(없으면 첫 Episode)를 골라 승격합니다. 승격에는 **비식별화 기록**이 필요합니다. 원자료에 보관하면 안 되는 정보가 없는지 직접 확인하고 아래 `SANITIZATION_NOTE`에 한두 문장으로 적으세요. 비워 두면 승격이 거부됩니다. `RUN_EXERCISE = True`로 켜야 실행됩니다.

In [ ]:
# [내 것 만들기] 해보기 3: Code 판정이 FAIL인 Episode 하나를 미검수 후보로 승격한다
SANITIZATION_NOTE = ""   # 직접 작성한다: 이 원자료에 개인정보나 보관 금지 정보가 없는지 확인한 내용

if not RUN_EXERCISE:
    print("RUN_EXERCISE = True로 바꾸면 승격을 시도합니다. 먼저 SANITIZATION_NOTE를 직접 적으세요.")
else:
    raw_recording = {
        "run_id": make_run_id(), "snapshot": frozen["snapshot"],
        "rows": [{"artifact": artifact.model_dump(), "trace_id": trace_by_key[episode_key(artifact)], "verdicts": []}
                 for artifact in artifacts],
    }
    target = next((a for a in artifacts if code_result[episode_key(a)] == "FAIL"), artifacts[0])
    try:
        promoted = promote_failure(raw_recording, {"artifact_id": target.artifact_id}, run_dir / "promoted.xlsx",
                                   sanitization_record=SANITIZATION_NOTE)
    except ValueError as error:
        print("승격 거부:", error)
    else:
        print("승격 완료: 미검수 후보", promoted["metadata"]["scenario_id"], "| 검수 상태", promoted["metadata"]["review_status"])
        try:
            Scenario.model_validate(promoted)
        except ValidationError as error:
            print("아직 계약을 통과하지 못합니다:", error.errors()[0]["loc"], "-> 담당자가 기대 상태를 채워야 합니다.")

**관찰할 것.** 비식별화 기록을 비우면 승격이 **거부**됩니다. 적으면 `promoted.xlsx`(검수용 엑셀)와 초기 상태 파일이 만들어지고, 카드의 기대 최종 상태는 **비어 있어서** 계약을 통과하지 못합니다. 담당자가 정책에 따라 채우기 전에는 평가에 쓰지 못한다는 뜻입니다. 이렇게 승격한 후보는 dev에 바로 합치지 않고 검수와 동결을 새로 거칩니다.

### 해보기 4. 내 문턱값 고르기

`MY_T_LO`와 `MY_T_HI`를 정해, 저장된 Jev 확률로 넘긴 비율을 보세요. `RUN_OPTIONAL = True`로 켜면 후보 격자 전체를 한 번에 봅니다. 어느 문턱값이 "좋은지"는 이 표로 정할 수 없고 사람 라벨 대비 false pass와 false reject로만 정할 수 있다는 점을 기억하세요.

In [ ]:
# [내 것 만들기] 해보기 4: 내 문턱값에서 Judge로 넘어갈 질문의 비율을 저장된 Jev 확률로 다시 센다
MY_T_LO, MY_T_HI = 0.3, 0.7   # 직접 바꾼다. 둘 다 미조정 값이다
print(f"내 문턱값 ({MY_T_LO}, {MY_T_HI}): 넘긴 비율 {escalation_rate_at(cascade, MY_T_LO, MY_T_HI):.2f}")
print(f"기본 자리값 ({T_LO}, {T_HI}): 넘긴 비율 {escalation_rate_at(cascade, T_LO, T_HI):.2f}")

if RUN_OPTIONAL:
    grid = [round(step / 10, 1) for step in range(10)]
    show([{"t_lo": lo, "t_hi": hi, "넘긴 비율": round(escalation_rate_at(cascade, lo, hi), 2)}
          for lo in grid for hi in grid if lo < hi])

**관찰할 것.** 구간이 넓을수록 넘긴 비율이 단조롭게 커집니다. 격자 표에서 "비용은 낮은데 라벨이 없는" 영역을 찾아 보세요. 그런 영역은 값싸 보이지만 사람 라벨 없이는 안전하다고 말할 수 없는 영역입니다.

## 확인 질문

1. Scenario에서 `expected_output`을 에이전트에게 주지 않는 이유는 무엇인가요? 에이전트의 실제 결과를 기대 상태로 복사하면 어떤 평가가 되는지 자기 말로 설명해 보세요.
2. 정책 문서에서 만든 후보를 모델이 정답까지 채우게 하지 않고 사람이 승인하도록 한 이유는 무엇인가요? 후보에 붙이는 두 가지 근거는 무엇인가요?
3. 최소쌍이 "문장을 바꾼 변형"과 다른 점은 무엇인가요? 조건이 둘 이상 바뀐 쌍에서 에이전트가 실패하면 어떤 점이 곤란한가요?
4. 동결 해시는 무엇을 막나요? 기준을 정당하게 고쳐야 할 때는 파일을 고치지 않고 무엇을 하나요?
5. `load_dataset(split="holdout")`이 거부되는 이유를 말해 보세요. `approved_only=True`로 읽은 결과를 "전체 holdout을 열었다"고 말할 수 없는 이유는 무엇인가요?
6. 연쇄 판정에서 Jev와 Judge는 각각 언제 쓰이나요? 문턱값 `0.2`와 `0.8`을 "미조정"이라고 부르는 이유는 무엇이고, 이 값을 어떻게 정해야 하나요?
7. 사람 라벨이 전부 PASS일 때 false pass가 0%라는 결과를 안전의 증거로 쓸 수 없는 이유는 무엇인가요? 이때 어떻게 보고해야 하나요?

## 정리

- 사례 카드는 입력(에이전트용), 기대 기준(평가자용), 꼬리표(추적용)로 나뉩니다. 기대 상태는 사람이 정책 원문과 대조해 정합니다.
- 사례는 인터뷰(기준), 문서(범위), 운영 trace(빠진 실패)의 세 경로로 얻고, 문서에서 만든 후보는 근거 문단과 정책 버전을 붙여 사람이 승인합니다.
- 경계 사례는 한 조건만 바꾼 최소쌍으로 만들고, 같은 사건의 가족은 dev와 holdout에 나눠 담지 않습니다.
- 승인된 사례는 해시와 함께 동결합니다. 기준을 고치려면 새 버전을 만듭니다. holdout은 담당자의 승인 전에는 엄격하게 열리지 않습니다.
- Episode는 Langfuse에서 root observation 하나이고, 실험은 run 이름과 version 라벨로 구분해 기록합니다. 키가 없으면 같은 내용을 로컬 파일에 남깁니다.
- 판정은 Code(사실), Jev(먼저), Judge(불확실할 때), Human(기준 수정)으로 나눕니다. 연쇄는 비용을 줄이지만 Jev의 확률은 보정되지 않았고 문턱값은 미조정입니다.
- 자동 판정의 신뢰는 사람 라벨 대비 false pass와 false reject로만 잽니다. 라벨이 한쪽뿐이면 `insufficient_coverage`입니다.

**미확인.** 실제 에이전트 모델, Jev, Judge의 응답. Langfuse 서버가 Dataset 항목의 시점별 버전과 run 연결을 어떻게 집계하고 보여 주는지. 이 수업에서는 서버 쪽 동작을 확인하지 않았습니다.

다음 03에서는 정답이 없는 운영 trace를 무엇으로, 어떤 것부터 채점하는지 봅니다.